# Day 1: API Data Extraction and Transformation

**Scenario.** Duka Connect, a Nairobi-based B2B distributor, is onboarding a partner whose customer
directory is only available through a REST API. This notebook pulls the customer records from the
API, works out the shape of the nested JSON, flattens it, and exports a clean CSV
(`output/customers_clean.csv`) that the analytics team can load straight into a spreadsheet.

**Source:** <https://jsonplaceholder.typicode.com/users>

Each of the eight tasks in the brief has its own section below. The functions keep the
signatures from `starter.py`; `main()` at the end wires them into one pipeline.

## Setup

In [1]:
# importing libraries
import os                  # read environment variables (used to override the API URL)
import sys                 # print errors to stderr and exit with a status code
from pathlib import Path   # build file paths that work on Windows, Mac and Linux

import pandas as pd        # tables (DataFrames) and CSV export
import requests            # HTTP requests to the API

# settings
# The URL can be overridden from the command line (see README) to test failure handling.
API_URL = os.environ.get("CUSTOMERS_API_URL", "https://jsonplaceholder.typicode.com/users")
OUTPUT_PATH = Path("output") / "customers_clean.csv"   # output/customers_clean.csv

# Column order taken from expected_customers_clean.csv
EXPECTED_COLUMNS = [
    "customer_id", "name", "username", "email", "phone", "website",
    "address_street", "address_suite", "address_city", "address_zipcode",
    "address_geo_lat", "address_geo_lng",
    "company_name", "company_catchphrase", "company_bs",
]

## Task 1: Retrieve the users with `requests.get()`

`fetch_users()` calls the endpoint with a timeout so a slow server cannot hang the script, and
checks the status code **before** touching the body. Every way the request can go wrong is turned
into a single `FetchError` with a plain-English message; Task 8 relies on this.
`fetch_users_or_exit()` is the version the pipeline uses: it prints that message and stops with
exit code 1 instead of showing a traceback.

`Timeout` is caught before `ConnectionError` because a connect timeout is a subclass of both.

In [2]:
# custom error type: every way the request can fail is turned into this one exception
class FetchError(Exception):
    """Raised when the API cannot give us a usable list of users."""


def fetch_users(url: str = API_URL, timeout: float = 10.0) -> list[dict]:
    """Retrieve the list of users from the API.

    Raises FetchError with a clear message for connection failures, timeouts,
    non-200 status codes and bodies that are not valid JSON.
    """
    # send the GET request; timeout stops the script hanging on a slow server
    try:
        response = requests.get(url, timeout=timeout)
    # Timeout is checked first because a connect timeout is also a ConnectionError
    except requests.exceptions.Timeout:
        raise FetchError(f"Request to {url} timed out after {timeout} seconds.") from None
    # wrong host name, no internet, server refused the connection
    except requests.exceptions.ConnectionError:
        raise FetchError(f"Could not connect to {url}. Check the URL and your internet connection.") from None
    # anything else requests can raise (e.g. a badly formed URL)
    except requests.exceptions.RequestException as exc:
        raise FetchError(f"Request to {url} failed: {exc}") from None
    # "from None" hides the original traceback so only our message is shown

    # confirm the status code before trusting the body
    if response.status_code != 200:
        raise FetchError(f"{url} returned HTTP {response.status_code} ({response.reason}), expected 200.")

    # parse the body as JSON; an HTML page or plain text raises ValueError
    try:
        payload = response.json()
    except ValueError:
        content_type = response.headers.get("Content-Type", "unknown")
        raise FetchError(f"{url} returned a body that is not valid JSON (Content-Type: {content_type}).") from None

    # the users endpoint should return a list; anything else is not usable
    if not isinstance(payload, list):
        raise FetchError(f"Expected a JSON list of users from {url}, got a {type(payload).__name__}.")

    return payload


def fetch_users_or_exit(url: str = API_URL) -> list[dict]:
    """Fetch the users, or print one clear error message and stop with exit code 1."""
    try:
        return fetch_users(url)
    except FetchError as exc:
        print(f"ERROR: {exc}", file=sys.stderr)   # one clear message, no traceback
        sys.exit(1)                               # non-zero exit = the run failed

In [3]:
# fetching the users from the API
users = fetch_users_or_exit()
print(f"Fetched {len(users)} users from {API_URL}")

Fetched 10 users from https://jsonplaceholder.typicode.com/users


## Task 2: Identify the returned data structures

`describe_structure()` prints the type of the top-level payload, of one record, and of every value
inside that record. It recurses into nested dicts, so it reports structure at any depth.

In [4]:
def describe_structure(payload) -> None:
    """Print what Python types the payload and one record are made of
    (the top-level container, a record, and each nested value)."""
    # the outer container (a list of users)
    print(f"Top-level payload: {type(payload).__name__} of {len(payload)} items")
    if not payload:   # nothing to inspect in an empty list
        return
    # one record is enough to show the shape, since every user has the same layout
    record = payload[0]
    print(f"One record: {type(record).__name__} with {len(record)} keys")
    _describe_values(record, depth=1)


# helper: prints every key and its type, indenting one level for each nested dict
def _describe_values(record: dict, depth: int) -> None:
    pad = "    " * depth   # indentation grows with the nesting depth
    for key, value in record.items():
        if isinstance(value, dict):
            # nested dict: report it, then call this function again on the inner dict
            print(f"{pad}{key}: dict with {len(value)} keys  (nesting depth {depth})")
            _describe_values(value, depth + 1)
        else:
            # plain value: show its type and the value itself
            print(f"{pad}{key}: {type(value).__name__} = {value!r}")


# inspecting the payload
describe_structure(users)

Top-level payload: list of 10 items
One record: dict with 8 keys
    id: int = 1
    name: str = 'Leanne Graham'
    username: str = 'Bret'
    email: str = 'Sincere@april.biz'
    address: dict with 5 keys  (nesting depth 1)
        street: str = 'Kulas Light'
        suite: str = 'Apt. 556'
        city: str = 'Gwenborough'
        zipcode: str = '92998-3874'
        geo: dict with 2 keys  (nesting depth 2)
            lat: str = '-37.3159'
            lng: str = '81.1496'
    phone: str = '1-770-736-8031 x56442'
    website: str = 'hildegard.org'
    company: dict with 3 keys  (nesting depth 1)
        name: str = 'Romaguera-Crona'
        catchPhrase: str = 'Multi-layered client-server neural-net'
        bs: str = 'harness real-time e-markets'


**What the structure tells us**

- The payload is a **list** of 10 **dicts**, one dict per customer.
- Most fields are flat strings (`name`, `username`, `email`, `phone`, `website`); `id` is an int.
- Nested fields:
  - `address` is a dict, **1 level** deep (`street`, `suite`, `city`, `zipcode`)
  - `address.geo` is a dict inside `address`, **2 levels** deep (`lat`, `lng`)
  - `company` is a dict, **1 level** deep (`name`, `catchPhrase`, `bs`)
- `lat` and `lng` arrive as **strings**, not numbers, so they must be converted in Task 6.

## Task 3: Extract names, emails, cities and companies

Nested lookups use `.get()` with `{}` as the fallback for a missing parent dict, so a record with
no `address` or `company` does not raise a `KeyError`.

**Default chosen:** a missing field becomes `None`. In pandas and in the CSV that is an empty cell,
which the analytics team can tell apart from a real value. A made-up placeholder like `"Unknown"`
could be mistaken for a real city.

In [5]:
def extract_core_fields(users: list[dict]) -> list[dict]:
    """Return one small dict per user with id, name, email, city and company name."""
    core = []
    # plain for loop over every user record
    for user in users:
        # .get() returns None instead of raising KeyError when a key is missing;
        # "or {}" swaps a missing address/company for an empty dict so the next .get() still works
        address = user.get("address") or {}
        company = user.get("company") or {}
        core.append({
            "id": user.get("id"),
            "name": user.get("name"),
            "email": user.get("email"),
            "city": address.get("city"),      # nested: address -> city
            "company": company.get("name"),   # nested: company -> name
        })
    return core


# extracting the core fields and previewing the first three
core = extract_core_fields(users)
core[:3]

[{'id': 1,
  'name': 'Leanne Graham',
  'email': 'Sincere@april.biz',
  'city': 'Gwenborough',
  'company': 'Romaguera-Crona'},
 {'id': 2,
  'name': 'Ervin Howell',
  'email': 'Shanna@melissa.tv',
  'city': 'Wisokyburgh',
  'company': 'Deckow-Crist'},
 {'id': 3,
  'name': 'Clementine Bauch',
  'email': 'Nathan@yesenia.net',
  'city': 'McKenziehaven',
  'company': 'Romaguera-Jacobson'}]

In [6]:
# testing a record with no address and no company: no crash, the gaps come back as None
extract_core_fields([{"id": 99, "name": "Wanjiru Traders", "email": "info@wanjiru.co.ke"}])

[{'id': 99,
  'name': 'Wanjiru Traders',
  'email': 'info@wanjiru.co.ke',
  'city': None,
  'company': None}]

## Task 4: Build lists, tuples and dicts with loops and comprehensions

- list of names: **list comprehension**
- list of `(id, email)` tuples: **list comprehension** producing tuples
- dict of id to city: **dict comprehension**

The plain `for` loop requirement is met in `extract_core_fields()` above (and in `flatten_user()` below).

In [7]:
def build_structures(users: list[dict]) -> tuple[list, list[tuple], dict]:
    """Return (list_of_names, list_of_(id, email)_tuples, dict_of_id_to_city)."""
    # list comprehension: one name per user
    names = [user.get("name") for user in users]
    # list comprehension that builds a tuple for each user
    id_email_pairs = [(user.get("id"), user.get("email")) for user in users]
    # dict comprehension: customer id as the key, city as the value
    id_to_city = {user.get("id"): (user.get("address") or {}).get("city") for user in users}
    return names, id_email_pairs, id_to_city


# building the three structures and printing them
names, id_email_pairs, id_to_city = build_structures(users)
print("Names:", names, "\n")
print("(id, email) pairs:", id_email_pairs[:3], "...\n")
print("id -> city:", id_to_city)

Names: ['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Patricia Lebsack', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Kurtis Weissnat', 'Nicholas Runolfsdottir V', 'Glenna Reichert', 'Clementina DuBuque'] 

(id, email) pairs: [(1, 'Sincere@april.biz'), (2, 'Shanna@melissa.tv'), (3, 'Nathan@yesenia.net')] ...

id -> city: {1: 'Gwenborough', 2: 'Wisokyburgh', 3: 'McKenziehaven', 4: 'South Elvis', 5: 'Roscoeview', 6: 'South Christy', 7: 'Howemouth', 8: 'Aliyaview', 9: 'Bartholomebury', 10: 'Lebsackbury'}


## Task 5: Flatten the nested JSON

`flatten_user()` is recursive: whenever a value is itself a dict it calls itself with the key so
far as the prefix. No field names are hard-coded, so it works for any depth and for records it has
never seen. `address -> geo -> lat` becomes `address_geo_lat`.

In [8]:
def flatten_user(user: dict, parent_key: str = "", sep: str = "_") -> dict:
    """Flatten one nested user record into a single-level dict.

    e.g. a nested address -> geo -> lat becomes the key address_geo_lat.
    """
    flat = {}
    for key, value in user.items():
        # join the parent's name onto this key (address + geo -> address_geo);
        # at the top level there is no parent, so the key stays as it is
        new_key = f"{parent_key}{sep}{key}" if parent_key else key
        if isinstance(value, dict):
            # nested dict: flatten it too (recursion), passing the joined key as its prefix
            flat.update(flatten_user(value, new_key, sep))
        else:
            # plain value: store it under the joined key
            flat[new_key] = value
    return flat


# flattening every user and previewing the first one
flat_users = [flatten_user(user) for user in users]
flat_users[0]

{'id': 1,
 'name': 'Leanne Graham',
 'username': 'Bret',
 'email': 'Sincere@april.biz',
 'address_street': 'Kulas Light',
 'address_suite': 'Apt. 556',
 'address_city': 'Gwenborough',
 'address_zipcode': '92998-3874',
 'address_geo_lat': '-37.3159',
 'address_geo_lng': '81.1496',
 'phone': '1-770-736-8031 x56442',
 'website': 'hildegard.org',
 'company_name': 'Romaguera-Crona',
 'company_catchPhrase': 'Multi-layered client-server neural-net',
 'company_bs': 'harness real-time e-markets'}

In [9]:
# testing a record shape it has never seen, three levels deep (no field names are hard-coded)
flatten_user({"id": 1, "branch": {"region": {"county": {"name": "Kiambu", "code": 22}}}})

{'id': 1,
 'branch_region_county_name': 'Kiambu',
 'branch_region_county_code': 22}

## Task 6: Load the records into a DataFrame

`to_dataframe()`:

1. builds the DataFrame and lower-cases the column names (`company_catchPhrase` becomes
   `company_catchphrase`, matching the expected output)
2. renames `id` to `customer_id`
3. puts the columns in the expected order; any column the API adds later that is not in the agreed
   schema is reported and left out, so the analytics team always gets the same layout
4. converts `address_geo_lat` / `address_geo_lng` from strings to floats
5. drops rows with a blank `customer_id` and duplicate customers, so there is one row per user

In [10]:
def to_dataframe(flat_users: list[dict]) -> pd.DataFrame:
    """Build a DataFrame with the columns in the expected-output order."""
    # one row per flattened user, one column per key
    df = pd.DataFrame(flat_users)

    # lower_snake_case column names (company_catchPhrase -> company_catchphrase)
    df.columns = [col.lower() for col in df.columns]
    df = df.rename(columns={"id": "customer_id"})

    # keep only the agreed columns, in the agreed order; report anything new from the API
    extra = [col for col in df.columns if col not in EXPECTED_COLUMNS]
    if extra:
        print(f"Note: leaving out columns not in the expected output: {extra}")
    df = df.reindex(columns=EXPECTED_COLUMNS)   # a missing expected column comes out empty

    # coordinates arrive as strings; convert to numbers ("coerce" turns bad values into NaN)
    for col in ["address_geo_lat", "address_geo_lng"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # one row per customer: drop blank and duplicate customer_ids
    rows_in = len(df)
    df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce")
    df = df.dropna(subset=["customer_id"]).drop_duplicates(subset="customer_id")
    if len(df) < rows_in:
        print(f"Note: removed {rows_in - len(df)} rows with a blank or duplicate customer_id")
    df["customer_id"] = df["customer_id"].astype(int)
    df = df.reset_index(drop=True)   # renumber rows 0..n after any drops

    # checking the result
    print(f"Shape: {df.shape}")
    print(df.dtypes)
    return df


# building the DataFrame and previewing the first five rows
df = to_dataframe(flat_users)
df.head()

Shape: (10, 15)
customer_id              int64
name                       str
username                   str
email                      str
phone                      str
website                    str
address_street             str
address_suite              str
address_city               str
address_zipcode            str
address_geo_lat        float64
address_geo_lng        float64
company_name               str
company_catchphrase        str
company_bs                 str
dtype: object


,customer_id,name,username,email,phone,website,address_street,address_suite,address_city,address_zipcode,address_geo_lat,address_geo_lng,company_name,company_catchphrase,company_bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems


## Task 7: Export the cleaned CSV

`to_csv()` opens the file in write mode by default, so re-running replaces the file rather than
appending to it. `index=False` keeps the pandas row index out of the CSV.

In [11]:
def export_csv(df: pd.DataFrame, path: Path = OUTPUT_PATH) -> Path:
    """Write the DataFrame to CSV (creating the folder if needed) and return the path."""
    path = Path(path)
    # create the output folder if it does not exist (no error if it already does)
    path.parent.mkdir(parents=True, exist_ok=True)
    # index=False leaves out the pandas row numbers; to_csv overwrites by default
    df.to_csv(path, index=False)
    return path


# exporting twice: the second run should overwrite the file, not append to it
saved = export_csv(df)
saved_again = export_csv(df)

# reading the file back to confirm rows, column order and number types
check = pd.read_csv(saved_again)
print(f"Saved to {saved_again.resolve()}")
print(f"Rows after exporting twice: {len(check)} (expected {len(df)})")
print(f"Columns match expected order: {list(check.columns) == EXPECTED_COLUMNS}")
print(check.dtypes[["customer_id", "address_geo_lat", "address_geo_lng"]])

Saved to C:\Users\Stanley\Desktop\Lux_Dev\API Data Extraction and Transformation\output\customers_clean.csv
Rows after exporting twice: 10 (expected 10)
Columns match expected order: True
customer_id          int64
address_geo_lat    float64
address_geo_lng    float64
dtype: object


## Task 8: Handle API request failures

Each failure below goes through `fetch_users()` and comes back as one clear `FetchError` message
instead of a raw traceback. In `main()` that message is printed and the run stops with exit code 1.

In [12]:
# failure test cases: (description, URL, timeout in seconds)
failure_cases = [
    ("Wrong URL (host does not exist)", "https://jsonplaceholder.typicode.invalid/users", 10),
    ("Non-200 response (404)", "https://jsonplaceholder.typicode.com/does-not-exist", 10),
    ("Body that is not JSON (HTML page)", "https://example.com", 10),
    ("Timeout", "https://jsonplaceholder.typicode.com/users", 0.001),   # far too short to succeed
]

# running each case: every one should raise FetchError with a clear message
for label, url, timeout in failure_cases:
    try:
        fetch_users(url, timeout=timeout)
        print(f"{label}: no error raised (unexpected)")
    except FetchError as exc:
        print(f"{label}:\n    {exc}")

Wrong URL (host does not exist):
    Could not connect to https://jsonplaceholder.typicode.invalid/users. Check the URL and your internet connection.


Non-200 response (404):
    https://jsonplaceholder.typicode.com/does-not-exist returned HTTP 404 (Not Found), expected 200.


Body that is not JSON (HTML page):
    https://example.com returned a body that is not valid JSON (Content-Type: text/html; charset=utf-8).
Timeout:
    Request to https://jsonplaceholder.typicode.com/users timed out after 0.001 seconds.


## Full pipeline

`main()` runs every step in order and prints a short summary. On an API failure it prints the
error message and exits with code 1. The README shows how this was tested from the command line.

In [13]:
def main() -> None:
    # 1. fetch (stops with a clear message and exit code 1 if the API fails)
    users = fetch_users_or_exit(API_URL)

    # 2 to 7. inspect, extract, build structures, flatten, load and export
    describe_structure(users)
    core = extract_core_fields(users)
    names, id_email_pairs, id_to_city = build_structures(users)
    df = to_dataframe([flatten_user(user) for user in users])
    path = export_csv(df)

    # short run summary
    print("\nRun summary")
    print(f"  Users fetched:         {len(users)}")
    print(f"  Core records built:    {len(core)}")
    print(f"  Unique cities:         {len(set(id_to_city.values()))}")
    print(f"  Rows x columns saved:  {df.shape[0]} x {df.shape[1]}")
    print(f"  Output file:           {path}")


# running the full pipeline
main()

Top-level payload: list of 10 items
One record: dict with 8 keys
    id: int = 1
    name: str = 'Leanne Graham'
    username: str = 'Bret'
    email: str = 'Sincere@april.biz'
    address: dict with 5 keys  (nesting depth 1)
        street: str = 'Kulas Light'
        suite: str = 'Apt. 556'
        city: str = 'Gwenborough'
        zipcode: str = '92998-3874'
        geo: dict with 2 keys  (nesting depth 2)
            lat: str = '-37.3159'
            lng: str = '81.1496'
    phone: str = '1-770-736-8031 x56442'
    website: str = 'hildegard.org'
    company: dict with 3 keys  (nesting depth 1)
        name: str = 'Romaguera-Crona'
        catchPhrase: str = 'Multi-layered client-server neural-net'
        bs: str = 'harness real-time e-markets'
Shape: (10, 15)
customer_id              int64
name                       str
username                   str
email                      str
phone                      str
website                    str
address_street             str
address_